# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2 Mass Spectrometry Extracellular Vesicles](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset via its Croissant schema
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print("Dataset Name: ", metadata.name)
print("Description: ", metadata.description)
print("Published: ", metadata.datePublished)
print("Identifier: ", metadata.identifier)
print("Keywords: ", getattr(metadata, 'keywords', None))

## 2. Data Overview
Review available record sets, fields, and their `@id` references.

We'll enumerate the available record sets in this dataset and their associated fields (columns). _All references are by their `@id` as per Croissant specification._

In [ ]:
# List all record sets and fields by their @id
print("Available Record Sets:")
record_sets = []
for rs in metadata.recordSets:
    record_sets.append(rs['@id'])
    print(f"- RecordSet @id: {rs['@id']}, name: {rs.get('name', '(no name)')}")
    print("    Fields/Columns:")
    for field in rs.get('fields', []):
        print(f"      - Field/Column @id: {field['@id']} name: {field.get('name', '')}")

Let's preview the first few records of each recordset. We use the exact `@id` string for each recordset.

In [ ]:
# Print a sample record from each available record set
for rs in metadata.recordSets:
    print(f"\nSample record from record set @id={rs['@id']}")
    records = dataset.records(record_set=rs['@id'])
    for i, rec in enumerate(records):
        print(rec)
        if i >= 1:  # Show only the first record or two
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

**Tip:** You may replace the chosen record set and field ids with any of those available in the output above for deeper exploration.

In [ ]:
# List all record set @ids for data extraction
record_set_ids = [rs['@id'] for rs in metadata.recordSets]
print("All record set @ids:", record_set_ids)

# Load records from all available recordsets into DataFrames
dataframes = dict()
for record_set_id in record_set_ids:
    df = pd.DataFrame(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = df
    print(f"RecordSet @id={record_set_id}: {df.shape[0]} rows, {df.shape[1]} columns")

# For demonstration, select the first record set for further analysis
main_record_set_id = record_set_ids[0]
main_df = dataframes[main_record_set_id]

print(f"\nFields (columns) in RecordSet {main_record_set_id}:")
print(list(main_df.columns))

# Show a few records
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we will:
- Select a numeric field (e.g., 'cr:field_Molecular_Weight' or as per the dataset semantics)
- Filter values that are above a certain threshold
- Normalize the selected field
- Group by a categorical field if available

**Update the field ids below as needed based on your exploration of columns above.**

In [ ]:
# Choose an example numeric field (replace this as appropriate for your use case)
# We'll use the first numeric-like field found
numeric_field_id = None
for col in main_df.columns:
    if main_df[col].dtype in ['float64', 'int64']:
        numeric_field_id = col
        break
# Fallback: forcibly pick a column that looks like molecular weight, MW, or a numeric field
if not numeric_field_id:
    for col in main_df.columns:
        if 'weight' in col.lower() or 'mw' in col.lower():
            numeric_field_id = col
            break

if numeric_field_id is None:
    raise ValueError("No numeric field found for analysis.")
print(f"Numeric field selected: {numeric_field_id}")

# Filter out records below a threshold
threshold = main_df[numeric_field_id].quantile(0.75) if main_df[numeric_field_id].dtype.kind in 'fi' else 0
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df[[numeric_field_id]].head())

# Normalize the numeric field (z-score)
normed_col = f"{numeric_field_id}_normalized"
filtered_df[normed_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, normed_col]].head())

# Group by a likely categorical field if available
group_field = None
for col in main_df.columns:
    if main_df[col].dtype == 'object' and main_df[col].nunique() < 10:
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean of {numeric_field_id} by {group_field}:")
    print(grouped_df.head())
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the selected numeric field
plt.figure(figsize=(8, 4))
sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.tight_layout()
plt.show()

# If a group field was found, plot the group means
if 'grouped_df' in locals() and group_field:
    plt.figure(figsize=(8,4))
    sns.barplot(x=group_field, y=numeric_field_id, data=grouped_df)
    plt.title(f"Mean of {numeric_field_id} by {group_field}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the [FAIR^2 Mass Spectrometry Vesicle Proteome](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset via its Croissant schema using `mlcroissant`. We listed all available record sets and fields by their `@id`, loaded records into pandas DataFrames, performed simple analytics such as numeric filtering, normalization, and group statistics, and visualized distributions in the dataset.

You can repeat or deepen this exploration for any record set, using the `@id` values for advanced queries.

_For more details about the dataset semantics and fields, refer back to the schema URL or associated publication._